# Split the Intellectual Property domain

Create a focused IP dataset for semantic search and archive the remaining articles without changing the original dataset.

In [ ]:
import json
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Could not find project root')

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
DOMAIN_DIR = DATA_DIR / 'processed'
ARCHIVE_DIR = PROJECT_ROOT / 'archive' / 'data' / 'non_intellectual_property'
DOMAIN_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

IP_DOCUMENT_IDS = {
    '11167', '11168', '11165', '11160', '1004', '1013',
    '1006', '11163', '11169', '11170', '11173',
}
records = [json.loads(line) for line in INPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
ip_records = [r for r in records if str(r.get('document_id')) in IP_DOCUMENT_IDS]
archive_records = [r for r in records if str(r.get('document_id')) not in IP_DOCUMENT_IDS]
print(f'All articles: {len(records)}')
print(f'IP articles: {len(ip_records)}')
print(f'Archived articles: {len(archive_records)}')

In [ ]:
def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows), encoding='utf-8')

ip_path = DOMAIN_DIR / 'articles_normalized.jsonl'
archive_path = ARCHIVE_DIR / 'articles_normalized.jsonl'
write_jsonl(ip_path, ip_records)
write_jsonl(archive_path, archive_records)
print(f'IP dataset: {ip_path}')
print(f'Archive dataset: {archive_path}')

In [ ]:
def document_summary(rows):
    grouped = {}
    for row in rows:
        document_id = str(row.get('document_id'))
        grouped.setdefault(document_id, {'document_id': document_id, 'title': row.get('document_title', ''), 'articles': 0})
        grouped[document_id]['articles'] += 1
    return sorted(grouped.values(), key=lambda item: item['document_id'])

for item in document_summary(ip_records):
    print(f"[{item['document_id']}] {item['title']} — {item['articles']} articles")